# Osoba B: GPT-2 + PEFT

Zakres (Część I, ~2,75 pkt): I.B, I.C, I.D, pkt 3 i 5 z [instrukcji](../Zadanie_2_zmodyfikowane.md).

## Do zrobienia

1. **I.B**: zapoznać się z biblioteką PEFT (quicktour + przykłady). Krótka notatka dla osoby A: jaki adapter (np. LoRA), jakie parametry, `task_type`.
2. **I.C**: powtórzyć tutorial z Medium (PEFT BERT-base, diagnozy medyczne) we własnym środowisku.
3. **I.D**: zapoznać się z klasami `GPT2Model`, `GPT2LMHeadModel`, `GPT2ForSequenceClassification`, `GPT2ForTokenClassification` (notatka w notebooku).
4. **Pkt 3: powtórzyć pkt 1 i 2 dla GPT-2**, model `sdadas/polish-gpt2-medium`, z adapterem PEFT:
   - klasyfikacja tekstu (`GPT2ForSequenceClassification`) i tokenów (`GPT2ForTokenClassification`), ocena na zbiorze testowym,
   - wizualizacja przestrzeni wektorowej dla tekstów testowych (tylko klasyfikacja tekstu), model bazowy vs dostrojony, dwa interaktywne wykresy + opis różnic.
5. **Pkt 5: augmentacja generacją GPT-2** (temperatura, top-p, top-k): porównać kilka ustawień, jaka jest jakość i różnorodność tekstów. Potem trening na zbiorze powiększonym.

## Ustalone (nie zmieniamy bez uzgodnienia z A)

- Model: `sdadas/polish-gpt2-medium`.
- Etykiety tekstu: `plus_m, minus_m, zero, amb` (`LABELS`, `LABEL2ID`). Tagi tokenów: BIO `wzmocnienie / oslabienie / negacja` (`TAGS`, `TAG2ID`).
- **Reprezentacja tekstu = ostatni token sekwencji** (ostatni token, który nie jest paddingiem), nie `[CLS]`. Dotyczy wizualizacji.
- **Maskowanie `-100`**: w klasyfikacji tokenów padding i znacznik `<eos>` dostają etykietę `-100`.
- Zbiór testowy jest wspólny z osobą A i nigdy nie jest augmentowany:
  - tekst: `load_text('test')`; zbiór treningowy: `load_text('train')`, walidacyjny: `load_text('dev')`,
  - tokeny: `split_tokens(load_tokens())` (seed 42).
- Augmentacja dotyczy tylko zbioru treningowego. Wynik zapisujemy w `../data/aug_gpt2.jsonl` (pola `text`, `label`, gdzie `label` to nazwa etykiety, a nie liczba). Finalny model trenujemy na `train + aug`.
- Adaptery i checkpointy zapisujemy w `../artifacts/`.

## Części wspólne (`../utils/`)

| Co | Skąd |
|---|---|
| wczytanie tekstów, etykiety | `utils.data`: `load_text`, `LABELS`, `LABEL2ID` |
| tagi BIO i podział train/test dla tokenów | `utils.tokens`: `load_tokens`, `split_tokens`, `TAGS`, `TAG2ID` |
| tagi słów → subtokeny (`-100`) | `utils.align`: `encode_tokens` |
| wykres przestrzeni wektorowej | `utils.plots`: `plot_embeddings` |

## Uwagi

- Tokenizer GPT-2: `AutoTokenizer.from_pretrained(MODEL, add_prefix_space=True)` (wymagane przy `is_split_into_words=True`) oraz `tokenizer.pad_token = tokenizer.eos_token` i `model.config.pad_token_id = tokenizer.pad_token_id`.
- `encode_tokens` oznacza `-100` tylko subtokeny niebędące pierwszym w słowie i pozycje specjalne. Jeśli tokenizer dopisuje `<eos>`, sprawdź, że dostał `-100` (`word_ids()` zwraca dla niego `None`).
- Wizualizacja: ostatni nie-paddingowy token (`attention_mask.sum(1) - 1`) z ostatniej warstwy ukrytej, przed głowicą. Dla `padding_side='right'` pamiętaj o tym indeksie.
- **Klasyfikacja tokenów ma mało danych**: anotacje z Zadania 1 to tylko 35 tekstów. Opisz to w wnioskach; rozszerzenie ustalamy z zespołem.
- Augmentacja: etykietę zachowaj przez prompt zawierający próbkę/początek tekstu z danej klasy. Odrzuć za krótkie i zduplikowane teksty.
- Metryki: tekst → accuracy i macro-F1 (`sklearn`); tokeny → precision/recall/F1 per tag, z pominięciem `-100`.

## Jak uruchamiać

Notebook ma trzy tryby, które przełącza komórka `parameters` poniżej (papermill ją nadpisuje):

| Tryb | Flagi | Dane i budżet | Czas (RTX 5060 8 GB) | Wynik |
|---|---|---|---|---|
| **pełny** | `PILOT=False` | cały train, 5 epok (tekst), 200 epok (tokeny), augmentacja 2 kopie | ok. 3–3,5 h | `gpt2_full.ipynb` |
| **pilot** | `PILOT=True` | 2000 tekstów train (warstwowo), pełny dev i test, 2 epoki, 50 epok (tokeny), augmentacja 1 kopia | ok. 30 min | `gpt2_pilot.ipynb` |
| smoke | `SMOKE=True` | kilkadziesiąt tekstów, 1 epoka: tylko test, że kod przechodzi | kilka min | nie do raportu |

```bash
cd 02/gpt2
papermill gpt2.ipynb gpt2_pilot.ipynb -p PILOT True --log-output
papermill gpt2.ipynb gpt2_full.ipynb  -p PILOT False --log-output
```

Każdy eksperyment zapisuje wynik w `../artifacts/gpt2-{tryb}/<eksperyment>/result.json` (i adapter w `best/`). Przy ponownym uruchomieniu wynik jest **wczytywany zamiast liczony**, więc przerwany run można wznowić tym samym poleceniem. Żeby policzyć coś od nowa, usuń odpowiedni katalog.

**Wyniki pilota są orientacyjne** (mniej danych i epok). Z BERT-em porównujemy dopiero pełny run.

In [ ]:
PILOT = False  # True: pilot (~30 min), False: pełny run (noc)
SMOKE = False  # True: tylko test techniczny (kilka minut)

In [ ]:
import os

# paski postępu (tqdm) zapisywałyby się jako widżety w wykonanym notebooku
os.environ.update(TQDM_DISABLE="1", HF_HUB_DISABLE_PROGRESS_BARS="1", HF_DATASETS_DISABLE_PROGRESS_BARS="1")
import sys

sys.path.insert(0, "..")
import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import torch
import datasets
from sklearn.model_selection import train_test_split
from transformers.utils import logging as hf_logging

from utils.align import encode_tokens
from utils.data import LABEL2ID, LABELS, load_aug, load_text
from utils.plots import plot_embeddings
from utils.tokens import TAG2ID, TAGS, load_tokens, split_tokens

MODEL = "sdadas/polish-gpt2-medium"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
MODE = "smoke" if SMOKE else "pilot" if PILOT else "full"
ART = Path("../artifacts") / f"gpt2-{MODE}"
ART.mkdir(parents=True, exist_ok=True)

CFG = {
    #        podpróbka train, dev/test,  epoki (tekst), rangi LoRA,      epoki (tokeny), prompty w siatce, kopie augmentacji
    "full": dict(n_train=None, n_eval=None, epochs=5, ranks=[0, 4, 8, 16], epochs_tok=200, n_grid=200, aug_copies=2),
    "pilot": dict(n_train=2000, n_eval=None, epochs=2, ranks=[0, 4, 8, 16], epochs_tok=50, n_grid=64, aug_copies=1),
    "smoke": dict(n_train=64, n_eval=32, epochs=1, ranks=[0, 8], epochs_tok=2, n_grid=8, aug_copies=1),
}[MODE]
T0 = time.time()
hf_logging.set_verbosity_error()  # bez raportów ładowania wag (MISSING/UNEXPECTED głowic)
hf_logging.disable_progress_bar()  # "Loading weights"
datasets.disable_progress_bars()  # "Map"
import warnings

warnings.filterwarnings("ignore", message="Could not find a config file")  # PEFT przy każdym zapisie w trybie offline


def log(msg):
    """Znacznik czasu od startu notebooka: widać postęp w logu papermilla."""
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def cached(name, fn):
    """Wynik eksperymentu `name` w ART/name/result.json. Jeśli plik istnieje, wczytujemy go zamiast liczyć."""
    path = ART / name / "result.json"
    if path.exists():
        log(f"{name}: wczytuję zapisany wynik ({path})")
        return json.loads(path.read_text())
    log(f"{name}: start")
    t0 = time.time()
    res = fn()
    res["wall_time_s"] = time.time() - t0
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(res, ensure_ascii=False, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o)))
    log(f"{name}: koniec ({res['wall_time_s'] / 60:.1f} min)")
    return res


def stratified(df, n):
    if n is None or n >= len(df):
        return df.reset_index(drop=True)
    return train_test_split(df, train_size=n, stratify=df["label"], random_state=SEED)[0].reset_index(drop=True)


train_full, dev, test = load_text("train"), load_text("dev"), load_text("test")
train = stratified(train_full, CFG["n_train"])
if CFG["n_eval"]:
    dev, test = stratified(dev, CFG["n_eval"]), stratified(test, CFG["n_eval"])
tok_train, tok_test = split_tokens(load_tokens())

print(f"TRYB: {MODE.upper()}  |  {DEVICE}  |  artefakty: {ART}")
print(CFG)
len(train), len(dev), len(test), len(tok_train), len(tok_test)

## I.B, I.C: PEFT (notatka)

I.C (tutorial z Medium, PEFT BERT-base na diagnozach medycznych) jest zrobione osobno (`artifacts/bert-lora-symptoms`).

**PEFT** (*Parameter-Efficient Fine-Tuning*): zamrażamy wagi modelu bazowego i trenujemy tylko mały zestaw nowych parametrów. Adapter zajmuje kilka MB zamiast całego modelu, mieści się w małej pamięci GPU i nie niszczy wiedzy z pretreningu.

**LoRA** (użyta tutaj i u BERT-a): dla wybranej macierzy wag $W \in \mathbb{R}^{d_{out} \times d_{in}}$ uczymy poprawkę niskiego rzędu
$$W' = W + \tfrac{\alpha}{r} B A, \quad A \in \mathbb{R}^{r \times d_{in}},\ B \in \mathbb{R}^{d_{out} \times r}.$$
$A$ jest losowe, a $B = 0$ na starcie, więc na początku $W' = W$. Trenujemy tylko $A$ i $B$. Po treningu `merge_and_unload()` wpisuje $W + \frac{\alpha}{r}BA$ w wagi, więc inferencja nic nie kosztuje.

| Parametr `LoraConfig` | U nas | Znaczenie |
|---|---|---|
| `r` | 4, 8, 16 | rząd poprawki: pojemność adaptera |
| `lora_alpha` | `2r` | skala poprawki ($\alpha/r = 2$ stałe, więc zmiana `r` nie zmienia skali) |
| `lora_dropout` | 0,1 | dropout na wejściu adaptera |
| `target_modules` | `c_attn` (GPT-2), `query`, `value` (BERT) | które macierze dostają adapter |
| `fan_in_fan_out` | `True` (GPT-2) | GPT-2 trzyma wagi w `Conv1D` jako $(d_{in}, d_{out})$, czyli transponowane względem `nn.Linear` |
| `task_type` | `SEQ_CLS`, `TOKEN_CLS` | PEFT sam dodaje głowicę (`score` / `classifier`) do `modules_to_save`, więc jest trenowana w całości |

Inne metody z PEFT: *prompt tuning* / *prefix tuning* (uczone wektory doklejane do wejścia lub do kluczy i wartości uwagi), *IA³* (uczone skalowanie aktywacji). LoRA jest najpopularniejsza, bo jest prosta i nie wydłuża sekwencji.

**Różnica GPT-2 vs BERT w adapterze:** w GPT-2 `c_attn` to jedna macierz $1024 \to 3072$, która liczy naraz Q, K i V. LoRA na `c_attn` adaptuje więc Q, K i V, a u BERT-a tylko Q i V. Na warstwę: GPT-2 ma $r(1024 + 3072) = 4096r$ parametrów adaptera, BERT $2 \cdot r(768 + 768) = 3072r$. Do tego GPT-2 ma 24 warstwy, a BERT 12, więc przy tym samym `r` adapter GPT-2 jest ok. 2,7× większy. Względem rozmiaru modelu (355M vs 124M) udział jest podobny: ok. 0,2%.

## I.D: klasy GPT-2 (notatka)

Wszystkie cztery klasy dzielą ciało `model.transformer` (`GPT2Model`). Wymiary dla `polish-gpt2-medium`: 24 bloki, `n_embd = 1024`, 16 głowic, słownik 51 200 tokenów (byte-level BPE), kontekst 2048.

**Najważniejsza różnica względem BERT-a: maska przyczynowa.** Token na pozycji $t$ widzi tylko tokeny $1..t$. Nie ma `[CLS]` ani poolera. Jedynym tokenem, który „widzi” cały tekst, jest **ostatni**, dlatego to on reprezentuje tekst.

**Wejścia:** `input_ids` `(B, T)` i `attention_mask` `(B, T)`. Brak `token_type_ids`. Tokenizer **nie ma tokenu paddingu**: ustawiamy `pad_token = eos_token` (`</s>`, id 2). Tokenizer sam nie dopisuje `</s>` ani `<s>`.

| Klasa | Głowica | `logits` / wyjście | `labels` → `loss` |
|---|---|---|---|
| `GPT2Model` | brak: `wte` + `wpe` → 24 bloki → `ln_f` | `last_hidden_state` `(B, T, 1024)` (już po `ln_f`), `past_key_values` (cache do generacji) | brak |
| `GPT2LMHeadModel` | `lm_head`: `Linear(1024 → 51200)`, bez biasu, związana z `wte` | `(B, T, 51200)`: rozkład **następnego** tokenu dla każdej pozycji | `(B, T)` = zwykle `input_ids`; przesunięcie o 1 robi model. CrossEntropy, `-100` pomijane |
| `GPT2ForSequenceClassification` | `score`: `Linear(1024 → num_labels)`, bez biasu, liczona na każdej pozycji, potem wybór **ostatniego tokenu różnego od `pad_token_id`** w każdym wierszu | `(B, 4)` | `(B,)`. Wymaga `config.pad_token_id`, inaczej nie obsłuży batcha > 1 |
| `GPT2ForTokenClassification` | dropout → `classifier`: `Linear(1024 → num_labels)` na każdej pozycji | `(B, T, 7)` | `(B, T)` z `-100` na paddingu, `</s>` i nie-pierwszych subtokenach słowa |

**Uwagi praktyczne:**
- `GPT2ForSequenceClassification` szuka ostatniego tokenu **różnego od `pad_token_id`**. Ponieważ `pad == eos`, dopisany na końcu `</s>` zostałby potraktowany jak padding. Dlatego w klasyfikacji tekstu nie dopisujemy `</s>`: głowica patrzy na ostatni prawdziwy token, a wizualizacja (3c) bierze ten sam token.
- W klasyfikacji tokenów (3b) `</s>` dopisujemy jawnie i maskujemy `-100`, zgodnie z instrukcją.
- Przy `is_split_into_words=True` tokenizer GPT-2 wymaga `add_prefix_space=True`: spacja należy do tokenu (byte-level BPE), więc słowo bez spacji przed sobą tokenizuje się inaczej.

In [ ]:
from transformers import (
    AutoTokenizer,
    GPT2ForSequenceClassification,
    GPT2ForTokenClassification,
    GPT2LMHeadModel,
    GPT2Model,
)

tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token  # GPT-2 nie ma paddingu: pad == eos (id 2)
batch = tok(["Hotel był bardzo czysty.", "Nie polecam."], padding=True, return_tensors="pt")
B, T = batch["input_ids"].shape
print("tokeny:", [[tok.decode([i]) for i in ids] for ids in batch["input_ids"]])  # spacja należy do tokenu

with torch.no_grad():
    out = GPT2Model.from_pretrained(MODEL)(**batch)
    assert out.last_hidden_state.shape == (B, T, 1024)
    for cls, kw, shape in [
        (GPT2LMHeadModel, {}, (B, T, 51200)),
        (GPT2ForSequenceClassification, {"num_labels": len(LABELS), "pad_token_id": tok.pad_token_id}, (B, len(LABELS))),
        (GPT2ForTokenClassification, {"num_labels": len(TAGS)}, (B, T, len(TAGS))),
    ]:
        m = cls.from_pretrained(MODEL, **kw).eval()
        assert m(**batch).logits.shape == shape
        head = m.lm_head if hasattr(m, "lm_head") else m.score if hasattr(m, "score") else m.classifier
        print(f"{cls.__name__:32} logits {tuple(shape)}  głowica: {head}")
        if cls is GPT2LMHeadModel:
            print(" " * 33, "lm_head związana z wte:", m.lm_head.weight.data_ptr() == m.transformer.wte.weight.data_ptr())
        del m
free_gpu()

## 3a. Klasyfikacja tekstu + PEFT

**Co badamy.** Czy `polish-gpt2-medium` z adapterem LoRA klasyfikuje sentyment recenzji (`plus_m, minus_m, zero, amb`) tak dobrze jak HerBERT. Protokół jest **1:1 z BERT-em (1a)**: rangi `r ∈ {4, 8, 16}` + baseline `r = 0` (zamrożony GPT-2, uczy się tylko głowica `score`), 5 epok, `lr = 2e-4`, `MAX_LEN = 400`, efektywny batch 16, checkpoint wybierany po macro-F1 na dev, ten sam test. Jedyna różnica techniczna: batch 4 × gradient accumulation 4, bo batch 16 nie mieści się w 8 GB (wynik matematycznie ten sam, Adam widzi średni gradient z 16 przykładów).

**Hipoteza.**
- LoRA: wynik zbliżony do HerBERT-a (test macro-F1 ok. 0,87). GPT-2 jest prawie 3× większy, ale przyczynowy: o klasie decyduje jeden ostatni token, który musi „streścić” cały tekst. HerBERT patrzy w obie strony i ma `[CLS]` trenowany do reprezentowania całości.
- Baseline `r=0` raczej **gorszy** niż u BERT-a (0,53). Ostatnia warstwa bazowego GPT-2 jest wyspecjalizowana w przewidywaniu *następnego tokenu*, a nie w streszczaniu tekstu. Liniowa głowica ma z niej mało do wyciągnięcia.
- `r` ma mały wpływ (tak było u BERT-a): już `r=4` daje dość pojemności dla 4 klas.

**Jak czytać.** Tabela: `dev_f1` (najlepsza epoka), `test_accuracy`, `test_f1` (macro, każda klasa waży tyle samo). Wykres krzywych: train loss, dev loss, dev F1 po epokach. Rosnący dev loss przy płaskim F1 oznacza przeuczenie (model staje się zbyt pewny).

**Pułapki.**
- Klasy są niezbalansowane (`minus_m` 38%, `zero` 15%), więc accuracy > F1. Patrzymy na macro-F1.
- `amb` (ambiwalentne) to najtrudniejsza klasa i zwykle ciągnie F1 w dół.
- 2,5% tekstów ma > 400 tokenów i jest ucinane: wtedy „ostatni token” leży w środku tekstu.
- **Pilot:** 2000 tekstów i 2 epoki, więc wyniki będą niższe niż w pełnym runie.

In [ ]:
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import (
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
)

from utils.train import make_dataset, make_token_dataset, plot_history, token_metrics
from utils.train import train as fit

BATCH, GRAD_ACCUM = 4, 4  # efektywny batch 16 jak u BERT-a
MAX_LEN, LR = 400, 2e-4


def make_model(r, auto=AutoModelForSequenceClassification, task=TaskType.SEQ_CLS, labels=LABELS):
    model = auto.from_pretrained(
        MODEL,
        num_labels=len(labels),
        id2label=dict(enumerate(labels)),
        label2id={l: i for i, l in enumerate(labels)},
        pad_token_id=tok.pad_token_id,
    )
    model.config.use_cache = False  # bez past_key_values w wyjściu (Trainer zbierałby je razem z logitami)
    cfg = LoraConfig(
        task_type=task,
        r=max(r, 1),
        lora_alpha=2 * max(r, 1),
        lora_dropout=0.1,
        target_modules=["c_attn"],
        fan_in_fan_out=True,  # Conv1D w GPT-2 trzyma wagi jako (in, out)
    )
    model = get_peft_model(model, cfg)  # głowica (`score` / `classifier`) trafia do modules_to_save
    if r == 0:  # baseline: B = 0 i zamrożone, więc ΔW = 0; uczy się tylko głowica
        for name, p in model.named_parameters():
            if "lora_" in name:
                p.requires_grad = False
    return model


def n_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


train_ds, dev_ds, test_ds = (make_dataset(d, tok, MAX_LEN) for d in (train, dev, test))


def run_seq(r, name, train_ds, epochs=CFG["epochs"]):
    def fn():
        model = make_model(r)
        trainable = n_trainable(model)
        trainer = fit(model, tok, train_ds, dev_ds, ART / name, epochs=epochs, lr=LR, batch_size=BATCH, grad_accum=GRAD_ACCUM)
        trainer.save_model(ART / name / "best")
        out = trainer.predict(test_ds)
        res = {
            "r": r,
            "trainable_params": trainable,
            "train_time_s": next(l["train_runtime"] for l in trainer.state.log_history if "train_runtime" in l),
            "dev_f1": trainer.state.best_metric,
            "test_accuracy": out.metrics["test_accuracy"],
            "test_f1": out.metrics["test_f1"],
            "test_pred": out.predictions.argmax(-1),
            "history": trainer.state.log_history,
        }
        del model, trainer
        free_gpu()
        return res

    return cached(name, fn)


seq = {r: run_seq(r, f"seq-r{r}", train_ds) for r in CFG["ranks"]}
name_of = lambda r: "baseline (r=0)" if r == 0 else f"LoRA r={r}"
results = pd.DataFrame(
    [{"model": name_of(r), **{k: v for k, v in res.items() if k not in ("test_pred", "history")}} for r, res in seq.items()]
)
results

In [ ]:
plot_history({name_of(r): res["history"] for r, res in seq.items()}, "3a. Krzywe uczenia: baseline vs GPT-2 + LoRA")

### 3aa. Porównanie na zbiorze testowym i macierz pomyłek

Macierz pomyłek dla najlepszego `r` (wg dev F1): wiersze to prawdziwa klasa, kolumny predykcja, wartości znormalizowane po wierszu (recall danej klasy na przekątnej).

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

px.bar(
    results.melt(id_vars="model", value_vars=["test_accuracy", "test_f1"], var_name="metryka", value_name="wynik"),
    x="model", y="wynik", color="metryka", barmode="group", text_auto=".3f",
    title="3a. Zbiór testowy: baseline (przed strojeniem) vs GPT-2 + LoRA",
).show()

best_r = max((r for r in seq if r > 0), key=lambda r: seq[r]["dev_f1"])
pred = np.array(seq[best_r]["test_pred"])
print(f"=== najlepsze r={best_r} (dev F1 {seq[best_r]['dev_f1']:.3f})")
print(classification_report(test["label_id"], pred, target_names=LABELS, digits=3))
cm = confusion_matrix(test["label_id"], pred, normalize="true")
px.imshow(cm, x=LABELS, y=LABELS, text_auto=".2f", color_continuous_scale="Blues",
          labels=dict(x="predykcja", y="prawda"), title=f"3a. Macierz pomyłek (LoRA r={best_r}, test)").show()

**Wnioski 3a** _(do uzupełnienia po pełnym runie)_:
- Ile daje LoRA względem baseline'u `r=0`? Czy baseline GPT-2 jest gorszy niż baseline BERT-a (0,53), i dlaczego?
- Czy `r` ma znaczenie (jakość, czas, liczba parametrów)?
- Czy krzywe pokazują przeuczenie (dev loss rośnie)?
- Które klasy są mylone (macierz pomyłek: `amb` vs `plus_m`/`minus_m`?)

## 3b. Klasyfikacja tokenów + PEFT (padding i `</s>` → `-100`)

**Co badamy.** Tagowanie BIO fraz `wzmocnienie`, `oslabienie`, `negacja` (7 tagów) przez `GPT2ForTokenClassification` + LoRA. Protokół jak u BERT-a (1b): te same dane (24 train / 4 dev / 7 test tekstów), `r ∈ {0, 4, 8, 16}`, 200 epok, `lr = 1e-3`, efektywny batch 8 (4 × 2). Metryka: macro-F1 po 6 tagach **bez `O`**, liczony na pierwszych subtokenach słów.

**Wymóg z instrukcji:** padding i `</s>` dostają etykietę `-100`. Tokenizer sam nie dopisuje `</s>`, więc dopisujemy go ręcznie na końcu każdej sekwencji (z etykietą `-100`). Padding dopełnia `DataCollatorForTokenClassification` (też `-100`). Poniżej sprawdzamy to assertami.

**Hipoteza.** GPT-2 będzie **gorszy lub co najwyżej porównywalny** z BERT-em (F1 0,24–0,39). Maska przyczynowa powoduje, że tag słowa jest przewidywany tylko z lewego kontekstu i z samego słowa:
- `B-negacja` („nie”) i typowe wzmocnienia („bardzo”) rozpoznaje się po samym słowie, więc powinny działać,
- granice fraz i tagi `I-` zależą od tego, co jest **dalej**, a tego GPT-2 nie widzi.

**Jak czytać.** Tabela i raport per tag (precision, recall, F1, `support` = liczba wystąpień w teście).

**Pułapki.**
- `O` to ok. 96% tokenów: accuracy ok. 0,97 osiąga sam „zawsze `O`”, dlatego F1 liczymy bez `O`.
- Test to 7 tekstów i ok. 30 tagów innych niż `O`. Jedna zmieniona predykcja przesuwa F1 o kilka punktów, a różnice między `r` są w granicach szumu.
- Tagi z `support = 0` w teście (`I-oslabienie`, `I-negacja`) mają F1 = 0 z definicji.

In [ ]:
EPOCHS_TOK, BATCH_TOK, GRAD_ACCUM_TOK, LR_TOK = CFG["epochs_tok"], 4, 2, 1e-3
MAX_LEN_TOK, N_DEV_TOK = 512, 4  # najdłuższy tekst: 418 tokenów GPT-2

tok_tc = AutoTokenizer.from_pretrained(MODEL, add_prefix_space=True)  # wymagane przy is_split_into_words=True
tok_tc.pad_token = tok_tc.eos_token
EOS = tok_tc.eos_token_id


def encode_tokens_eos(tokens, tags):
    """`encode_tokens` (tag na 1. subtokenie słowa, reszta -100) + `</s>` na końcu z etykietą -100."""
    enc = encode_tokens(tok_tc, tokens, tags, TAG2ID, MAX_LEN_TOK - 1)
    return {
        "input_ids": enc["input_ids"] + [EOS],
        "attention_mask": enc["attention_mask"] + [1],
        "labels": enc["labels"] + [-100],
    }


def make_token_ds(df):
    from datasets import Dataset
    return Dataset.from_list([encode_tokens_eos(t, g) for t, g in zip(df["tokens"], df["tags"])])


tok_tr, tok_dev = train_test_split(tok_train, test_size=N_DEV_TOK, random_state=42)
tok_train_ds, tok_dev_ds, tok_test_ds = (make_token_ds(d) for d in (tok_tr, tok_dev, tok_test))
collator_tok = DataCollatorForTokenClassification(tok_tc)

# --- sprawdzenie maskowania -100 na batchu z paddingiem
b = collator_tok([tok_train_ds[i] for i in range(4)])
ids, lab, att = b["input_ids"], b["labels"], b["attention_mask"]
assert (lab[att == 0] == -100).all(), "padding musi mieć -100"
assert (lab[ids == EOS] == -100).all(), "</s> (i padding, bo pad == eos) musi mieć -100"
last = att.sum(1) - 1
assert (ids[torch.arange(len(ids)), last] == EOS).all(), "ostatni prawdziwy token to </s>"
for i, (t, row) in enumerate(zip(tok_tr["tokens"][:4], lab)):
    assert (row != -100).sum() == len(t), "jedna etykieta na słowo"
print(f"OK: padding i </s> mają -100; batch {tuple(ids.shape)}, długości {att.sum(1).tolist()}")
print("koniec 1. przykładu:", list(zip([tok_tc.decode([i]) for i in ids[0, last[0] - 3 : last[0] + 2]], lab[0, last[0] - 3 : last[0] + 2].tolist())))
len(tok_train_ds), len(tok_dev_ds), len(tok_test_ds)

In [ ]:
def run_tok(r):
    def fn():
        model = make_model(r, AutoModelForTokenClassification, TaskType.TOKEN_CLS, TAGS)
        trainable = n_trainable(model)
        trainer = fit(model, tok_tc, tok_train_ds, tok_dev_ds, ART / f"tok-r{r}", epochs=EPOCHS_TOK, lr=LR_TOK,
                      batch_size=BATCH_TOK, grad_accum=GRAD_ACCUM_TOK, collator=collator_tok, compute_metrics=token_metrics)
        trainer.save_model(ART / f"tok-r{r}" / "best")
        out = trainer.predict(tok_test_ds)
        mask = out.label_ids != -100
        res = {
            "r": r,
            "trainable_params": trainable,
            "train_time_s": next(l["train_runtime"] for l in trainer.state.log_history if "train_runtime" in l),
            "dev_f1": trainer.state.best_metric,
            "test_accuracy": out.metrics["test_accuracy"],
            "test_f1": out.metrics["test_f1"],
            "test_labels": out.label_ids[mask],
            "test_pred": out.predictions.argmax(-1)[mask],
            "history": trainer.state.log_history,
        }
        del model, trainer
        free_gpu()
        return res

    return cached(f"tok-r{r}", fn)


tokres = {r: run_tok(r) for r in CFG["ranks"]}
results_tok = pd.DataFrame(
    [{"model": name_of(r), **{k: v for k, v in res.items() if k not in ("test_pred", "test_labels", "history")}} for r, res in tokres.items()]
)
results_tok

In [ ]:
plot_history({name_of(r): res["history"] for r, res in tokres.items()}, "3b. Krzywe uczenia (dev macro-F1 bez O)").show()
px.bar(
    results_tok.melt(id_vars="model", value_vars=["test_accuracy", "test_f1"], var_name="metryka", value_name="wynik"),
    x="model", y="wynik", color="metryka", barmode="group", text_auto=".3f",
    title="3b. Zbiór testowy: accuracy (z O) i macro-F1 (bez O)",
).show()
for r, res in tokres.items():
    print(f"=== {name_of(r)}")
    print(classification_report(res["test_labels"], res["test_pred"], labels=range(len(TAGS)), target_names=TAGS, digits=3, zero_division=0))

**Wnioski 3b** _(do uzupełnienia po pełnym runie)_:
- Czy model wychodzi poza przewidywanie samego `O`? Które tagi rozpoznaje (`B-negacja`, `B-wzmocnienie`?), a których nie?
- Jak wypada względem HerBERT-a (F1 0,24–0,39) i czy da się to powiązać z maską przyczynową (tagi `I-`, granice fraz)?
- Czy różnice między `r` są większe niż szum przy 7 tekstach testowych?

## 3c. Wizualizacja przestrzeni wektorowej (ostatni token): przed i po strojeniu

**Co badamy.** Jak teksty testowe układają się w przestrzeni reprezentacji GPT-2 względem etykiet, przed strojeniem i po nim. Reprezentacja tekstu to wektor **ostatniego prawdziwego tokenu** (`attention_mask.sum(1) - 1`, padding z prawej) z ostatniej warstwy (`last_hidden_state`, już po `ln_f`), czyli dokładnie to wejście, na które patrzy głowica `score`.
- **przed:** bazowy `GPT2Model`,
- **po:** GPT-2 z adapterem z 3a (`r = 4, 8, 16`) po `merge_and_unload()`.

Silhouette (cosinus) liczymy w pełnej przestrzeni 1024D. Bliżej 1 oznacza zwarte i rozdzielone klasy, ok. 0 klasy wymieszane, a wartość ujemna, że punkty są bliżej obcych klas niż swojej. Wykresy to t-SNE do 2D.

**Sprawdzenie poprawności:** dla modelu po strojeniu `score(reprezentacja)` musi dać te same logity co cały model, czyli wybieramy ten sam token co głowica. Dla `r=0` reprezentacja musi być identyczna z „przed” (ΔW = 0).

**Hipoteza.**
- **Przed:** silhouette ok. 0 lub ujemne. Ostatni token bazowego GPT-2 koduje głównie *co będzie dalej*, czyli zależy od ostatniego słowa i interpunkcji (np. teksty kończące się kropką lądują razem), a nie od sentymentu. Część II pokazała też silną anizotropię ostatnich warstw GPT-2.
- **Po:** wyraźne skupiska klas, silhouette podobne do BERT-a (ok. 0,5). Strojenie „przepisuje” ostatni token na streszczenie sentymentu. Kolejność klas na mapie (`minus_m` → `amb` → `plus_m`, osobno `zero`) powinna być taka sama jak u BERT-a.

**Jak czytać.** Kolor to prawdziwa etykieta. Po najechaniu widać tekst. Porównujemy strukturę (czy klasy tworzą skupiska, które się mieszają), a nie współrzędne: osie t-SNE są liczone osobno dla każdego wykresu.

**Pułapki.**
- t-SNE zachowuje sąsiedztwo, ale **odległości między skupiskami nic nie znaczą**. Ilościowo patrzymy na silhouette w 1024D.
- W ostatniej warstwie GPT-2 kilka wymiarów ma ogromne wartości i dominuje cosinus („rogue dimensions”). Silhouette dla „przed” może przez to być mało informatywne.

In [ ]:
from sklearn.metrics import silhouette_score


@torch.no_grad()
def last_token_embeddings(gpt2, texts, bs=16):
    """Ostatni nie-paddingowy token z last_hidden_state (po ln_f) -> (len(texts), 1024)."""
    gpt2.eval().to(DEVICE)
    out = []
    for i in range(0, len(texts), bs):
        b = tok(texts[i : i + bs], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(DEVICE)
        h = gpt2(**b).last_hidden_state
        last = b["attention_mask"].sum(1) - 1
        out.append(h[torch.arange(len(last)), last].float().cpu())
    return torch.cat(out).numpy()


def load_lora_seq(r, name=None):
    """GPT-2 + adapter z 3a, scalony w zwykły GPT2ForSequenceClassification."""
    base = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=len(LABELS), pad_token_id=tok.pad_token_id)
    return PeftModel.from_pretrained(base, ART / (name or f"seq-r{r}") / "best").merge_and_unload().eval().to(DEVICE)


def run_embeddings():
    texts = list(test["text"])
    embs = {"przed: GPT-2": last_token_embeddings(GPT2Model.from_pretrained(MODEL), texts)}
    free_gpu()
    m0 = load_lora_seq(0)
    assert np.allclose(last_token_embeddings(m0.transformer, texts[:16]), embs["przed: GPT-2"][:16], atol=1e-3), "r=0 musi mieć ΔW = 0"
    del m0
    for r in [r for r in CFG["ranks"] if r > 0]:
        m = load_lora_seq(r)
        e = last_token_embeddings(m.transformer, texts)
        with torch.no_grad():  # głowica na naszej reprezentacji == logity całego modelu
            b = tok(texts[:16], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(DEVICE)
            assert torch.allclose(m.score(torch.tensor(e[:16], device=DEVICE)), m(**b).logits, atol=1e-3)
        embs[f"po: LoRA r={r}"] = e
        del m
        free_gpu()
    return embs


emb_path = ART / "embeddings.npz"
if emb_path.exists():
    embs = dict(np.load(emb_path))
    log("3c: wczytuję zapisane reprezentacje")
else:
    log("3c: liczę reprezentacje")
    embs = run_embeddings()
    np.savez(emb_path, **embs)

silhouette = pd.DataFrame(
    [{"model": name, "silhouette (cosine, 1024D)": silhouette_score(e, test["label"], metric="cosine")} for name, e in embs.items()]
)
silhouette

In [ ]:
for name, e in embs.items():
    plot_embeddings(e, test["label"], list(test["text"]), title=f"3c. {name}: t-SNE wektorów ostatniego tokenu (test)", method="tsne").show()

**Wnioski 3c** _(do uzupełnienia po pełnym runie)_:
- Jak wyglądają klasy przed strojeniem? Czy widać jakąkolwiek strukturę (i od czego zależy: sentyment, ostatnie słowo, długość?)
- Jak po strojeniu? Gdzie leżą `amb` i `zero`? Jak zmienia się silhouette z `r`?
- Porównanie z BERT-em (silhouette przed ok. 0, po ok. 0,47–0,51).

## 5. Augmentacja generacją GPT-2 (temperatura, top-p, top-k)

**Co badamy.** Czy teksty wygenerowane przez bazowy GPT-2 powiększają zbiór treningowy na tyle dobrze, że klasyfikator się poprawia, i jak hiperparametry próbkowania wpływają na jakość i różnorodność tych tekstów.

**Jak generujemy (zachowanie etykiety).** GPT-2 bazowy nie wykonuje instrukcji, więc używamy **kontynuacji prefiksu**. Bierzemy tekst treningowy klasy *c*, zostawiamy pierwsze ok. 25% tokenów (min 10, max 40), a model dopisuje resztę do mniej więcej oryginalnej długości (30–300 nowych tokenów). Nowy tekst = prefiks + kontynuacja, z etykietą *c*. Zakładamy, że początek recenzji „ustawia” jej ton.

**Hiperparametry próbkowania.** Na każdym kroku model daje rozkład po słowniku. Temperatura $T$ dzieli logity przed softmaxem: $T < 1$ wyostrza rozkład (bezpieczniej, ale nudno, z pętlami), $T > 1$ go spłaszcza (różnorodnie, ale chaotycznie). **top-k** losuje tylko z $k$ najbardziej prawdopodobnych tokenów. **top-p** (nucleus) losuje z najmniejszego zbioru tokenów o łącznym prawdopodobieństwie ≥ $p$, więc liczba kandydatów dopasowuje się do pewności modelu.

| Ustawienie | Parametry | Oczekiwanie |
|---|---|---|
| A | `T=0.7, top_k=50` | płynnie i zachowawczo, ryzyko powtórzeń |
| B | `T=1.0, top_p=0.9` | standard: kompromis |
| C | `T=1.0, top_k=10` | bezpiecznie, ale mniej różnorodnie niż B |
| D | `T=1.3, top_p=0.95` | bardzo różnorodnie, gorsza płynność, dryf tematu |
| E | `T=1.0`, bez obcinania | kontrola: czyste próbkowanie, długi ogon dziwnych tokenów |

**Miary (na tej samej próbce promptów dla każdego ustawienia).**
- **distinct-1 / distinct-2:** liczba unikalnych 1-/2-gramów słów / liczba wszystkich w kontynuacjach (różnorodność, im więcej, tym bardziej różne teksty).
- **rep-4:** średni odsetek powtórzonych 4-gramów wewnątrz tekstu (pętle typu „bardzo bardzo bardzo…”, im mniej, tym lepiej).
- **perplexity:** perplexity kontynuacji pod bazowym GPT-2, warunkowana prefiksem (płynność, im mniej, tym bardziej „typowy” tekst). Odniesienie: perplexity prawdziwej dalszej części tych samych tekstów.
- **zgodność etykiety:** odsetek wygenerowanych tekstów, którym klasyfikator z 3a (`r=8`) przypisuje etykietę promptu. Odniesienia: ten sam klasyfikator na **samym prefiksie** (ile informacji o klasie daje już prefiks) i na **oryginalnym tekście** (górna granica; klasyfikator widział te teksty w treningu, więc jest zawyżona).

**Wybór ustawienia (automatyczny, bo pełny run idzie w nocy), w trzech krokach:**
1. **płynność:** odrzucamy ustawienia z perplexity > 2× perplexity prawdziwego tekstu (bełkot),
2. **różnorodność:** z pozostałych zostawiamy te z distinct-2 ≥ mediana (odrzucamy powtarzanie bezpiecznych fraz),
3. wybieramy **najwyższą zgodność etykiety**.

Krok 1 dodaliśmy po pilocie. Bez niego reguła wybrała D (`T=1.3`): perplexity 456 przy 36 dla prawdziwego tekstu, czyli bełkot. D miało i tak wysoką zgodność, bo klasyfikator ocenia głównie po prefiksie (zgodność na samym prefiksie: 0,64), a bełkotliwa kontynuacja tego nie zmienia. Sama zgodność i różnorodność nie wystarczą: obie da się „oszukać” chaosem.

**Augmentacja i trening.** Wybranym ustawieniem generujemy kopie każdego tekstu treningowego (pełny run: **2 kopie**, jak `[MASK]` u BERT-a; pilot: 1). Odrzucamy kontynuacje krótsze niż 20 tokenów i duplikaty. **Nie** filtrujemy klasyfikatorem, bo zbiór stałby się echem klasyfikatora. Zgodność etykiety tylko raportujemy. Wynik zapisujemy w `../data/aug_gpt2.jsonl` (pilot: `aug_gpt2_pilot.jsonl`). Trenujemy LoRA `r=8` na `train + aug` i porównujemy z `r=8` z 3a na tym samym teście.

**Hipoteza.**
- A i C: niska perplexity, wysoka zgodność, ale niska różnorodność i powtórzenia. D i E: odwrotnie.
- Wygra prawdopodobnie B lub C.
- Zgodność etykiety wyraźnie poniżej 1: model „zapomina” ton recenzji, zwłaszcza dla `amb` i `zero`.
- Zysk z augmentacji mały (u BERT-a +0,007 F1), a może nawet strata, jeśli dużo tekstów zmieniło sentyment. Szum w etykietach działa przeciwko nam.

**Pułapki.**
- Perplexity liczy ten sam model, który generował, więc niska temperatura daje niską perplexity „z definicji”. To miara typowości, a nie jakości.
- Zgodność etykiety mierzy klasyfikator, który sam się myli (ok. 12% na teście).
- Teksty z kontynuacji mają prefiks z prawdziwego tekstu, więc są blisko oryginałów (to bardziej parafraza niż nowy przykład).

In [ ]:
from transformers import AutoModelForCausalLM

R_AUG = 8
GEN_SETTINGS = {
    "A: T=0.7, top-k=50": dict(temperature=0.7, top_k=50, top_p=1.0),
    "B: T=1.0, top-p=0.9": dict(temperature=1.0, top_k=0, top_p=0.9),
    "C: T=1.0, top-k=10": dict(temperature=1.0, top_k=10, top_p=1.0),
    "D: T=1.3, top-p=0.95": dict(temperature=1.3, top_k=0, top_p=0.95),
    "E: T=1.0, bez obcinania": dict(temperature=1.0, top_k=0, top_p=1.0),
}  # top_k=0 jawnie: domyślnie HF dokłada top_k=50
GEN_BATCH, MIN_CONT = 32, 20
AUG_NAME = {"full": "gpt2", "pilot": "gpt2_pilot", "smoke": "gpt2_smoke"}[MODE]
AUG_PATH = Path(f"../data/aug_{AUG_NAME}.jsonl")


def make_prompts(df):
    """Prefiks = pierwsze ~25% tokenów (10-40), kontynuacja do ~oryginalnej długości (30-300 tokenów)."""
    rows = []
    for text, label in zip(df["text"], df["label"]):
        ids = tok(text).input_ids
        k = min(int(np.clip(round(0.25 * len(ids)), 10, 40)), len(ids))
        rows.append(dict(prefix_ids=ids[:k], rest_ids=ids[k:], label=label, src=text, n_new=int(np.clip(len(ids) - k, 30, 300))))
    return rows


@torch.no_grad()
def generate(lm, prompts, setting, seed):
    """Batche posortowane po długości, padding z lewej. Zwraca listę słowników z tekstem i statystykami."""
    torch.manual_seed(seed)
    order = sorted(range(len(prompts)), key=lambda i: prompts[i]["n_new"])
    out = [None] * len(prompts)
    for b in range(0, len(order), GEN_BATCH):
        idx = order[b : b + GEN_BATCH]
        L = max(len(prompts[i]["prefix_ids"]) for i in idx)
        ids = torch.full((len(idx), L), EOS)
        att = torch.zeros_like(ids)
        for j, i in enumerate(idx):
            p = prompts[i]["prefix_ids"]
            ids[j, L - len(p) :], att[j, L - len(p) :] = torch.tensor(p), 1
        gen = lm.generate(
            input_ids=ids.to(DEVICE), attention_mask=att.to(DEVICE), do_sample=True,
            max_new_tokens=max(prompts[i]["n_new"] for i in idx), pad_token_id=EOS, eos_token_id=EOS, **setting,
        )[:, L:].tolist()
        for j, i in enumerate(idx):
            cont = gen[j][: gen[j].index(EOS)] if EOS in gen[j] else gen[j]
            cont = cont[: prompts[i]["n_new"]]
            p = prompts[i]
            out[i] = dict(text=tok.decode(p["prefix_ids"] + cont).strip(), prefix=tok.decode(p["prefix_ids"]),
                          continuation=tok.decode(cont), cont_ids=cont, n_cont=len(cont), label=p["label"])
    return out


def distinct(texts, n):
    grams = [tuple(w[i : i + n]) for t in texts for w in [t.lower().split()] for i in range(len(w) - n + 1)]
    return len(set(grams)) / max(len(grams), 1)


def rep4(text):
    w = text.lower().split()
    grams = [tuple(w[i : i + 4]) for i in range(len(w) - 3)]
    return 1 - len(set(grams)) / len(grams) if grams else 0.0


@torch.no_grad()
def perplexity(lm, prefix_ids, cont_ids, bs=8):
    """exp(średnia NLL) tokenów kontynuacji pod `lm`, warunkowana prefiksem (prefiks i padding: -100)."""
    nll, n = 0.0, 0
    for b in range(0, len(prefix_ids), bs):
        seqs = [p + c for p, c in zip(prefix_ids[b : b + bs], cont_ids[b : b + bs])]
        L = max(map(len, seqs))
        ids = torch.full((len(seqs), L), EOS)
        lab = torch.full((len(seqs), L), -100)
        att = torch.zeros_like(ids)
        for j, (s, p) in enumerate(zip(seqs, prefix_ids[b : b + bs])):
            ids[j, : len(s)], att[j, : len(s)] = torch.tensor(s), 1
            lab[j, len(p) : len(s)] = torch.tensor(s[len(p) :])
        logits = lm(input_ids=ids.to(DEVICE), attention_mask=att.to(DEVICE)).logits.float()
        lp = torch.nn.functional.cross_entropy(logits[:, :-1].transpose(1, 2), lab[:, 1:].to(DEVICE), ignore_index=-100, reduction="sum")
        nll, n = nll + lp.item(), n + (lab[:, 1:] != -100).sum().item()
    return float(np.exp(nll / max(n, 1)))


@torch.no_grad()
def classify(clf, texts, bs=16):
    preds = []
    for i in range(0, len(texts), bs):
        b = tok(texts[i : i + bs], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(DEVICE)
        with torch.autocast(DEVICE, dtype=torch.bfloat16, enabled=DEVICE == "cuda"):
            preds += clf(**b).logits.argmax(-1).tolist()
    return np.array([LABELS[p] for p in preds])


lm = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).to(DEVICE).eval()
clf = load_lora_seq(R_AUG).to(DEVICE)  # klasyfikator z 3a do mierzenia zgodności etykiety

### 5a. Siatka ustawień generacji

In [ ]:
def run_grid():
    grid_df = stratified(train, CFG["n_grid"])
    prompts = make_prompts(grid_df)
    labels = np.array([p["label"] for p in prompts])
    ref = {
        "ustawienie": "odniesienie: prawdziwy tekst",
        "distinct-1": distinct([tok.decode(p["rest_ids"]) for p in prompts], 1),
        "distinct-2": distinct([tok.decode(p["rest_ids"]) for p in prompts], 2),
        "rep-4": float(np.mean([rep4(tok.decode(p["rest_ids"])) for p in prompts])),
        "perplexity": perplexity(lm, [p["prefix_ids"] for p in prompts], [p["rest_ids"][: p["n_new"]] for p in prompts]),
        "zgodność etykiety": float((classify(clf, [p["src"] for p in prompts]) == labels).mean()),
        "śr. długość kontynuacji": float(np.mean([min(len(p["rest_ids"]), p["n_new"]) for p in prompts])),
    }
    prefix_agree = float((classify(clf, [tok.decode(p["prefix_ids"]) for p in prompts]) == labels).mean())
    rows, examples = [], {}
    for name, setting in GEN_SETTINGS.items():
        log(f"5a: {name}")
        gen = generate(lm, prompts, setting, seed=SEED)
        conts = [g["continuation"] for g in gen]
        rows.append({
            "ustawienie": name,
            "distinct-1": distinct(conts, 1),
            "distinct-2": distinct(conts, 2),
            "rep-4": float(np.mean([rep4(c) for c in conts])),
            "perplexity": perplexity(lm, [p["prefix_ids"] for p in prompts], [g["cont_ids"] for g in gen]),
            "zgodność etykiety": float((classify(clf, [g["text"] for g in gen]) == labels).mean()),
            "śr. długość kontynuacji": float(np.mean([g["n_cont"] for g in gen])),
        })
        examples[name] = [{k: g[k] for k in ("label", "prefix", "continuation")} for g in gen[:3]]
    return {"grid": rows, "reference": ref, "prefix_agree": prefix_agree, "examples": examples}


grid_res = cached("aug-grid", run_grid)
grid = pd.DataFrame(grid_res["grid"]).set_index("ustawienie")
MAX_PPL = 2 * grid_res["reference"]["perplexity"]  # płynność: najwyżej 2x perplexity prawdziwego tekstu
fluent = grid[grid["perplexity"] <= MAX_PPL]
eligible = fluent[fluent["distinct-2"] >= fluent["distinct-2"].median()]
BEST_SETTING = eligible["zgodność etykiety"].idxmax()
print(f"Zgodność etykiety na samym prefiksie: {grid_res['prefix_agree']:.3f}")
print(f"Płynne (perplexity <= {MAX_PPL:.1f}): {list(fluent.index)}")
print(f"Płynne i różnorodne (distinct-2 >= {fluent['distinct-2'].median():.3f}): {list(eligible.index)}")
print(f"Wybrane ustawienie: {BEST_SETTING}")
pd.concat([grid, pd.DataFrame([grid_res["reference"]]).set_index("ustawienie")]).round(3)

In [ ]:
long = grid.reset_index().melt(id_vars="ustawienie", value_vars=["distinct-2", "rep-4", "perplexity", "zgodność etykiety"], var_name="miara", value_name="wartość")
px.bar(long, x="ustawienie", y="wartość", color="ustawienie", facet_col="miara", facet_col_wrap=4, text_auto=".2f",
       title="5a. Siatka generacji: różnorodność, powtórzenia, płynność, zgodność etykiety").update_yaxes(matches=None, showticklabels=True).update_xaxes(showticklabels=False).show()
px.scatter(grid.reset_index(), x="distinct-2", y="zgodność etykiety", text="ustawienie", size="perplexity",
           title="5a. Kompromis: różnorodność vs zachowanie etykiety (rozmiar = perplexity)").update_traces(textposition="top center").show()

for name, exs in grid_res["examples"].items():
    print(f"\n######## {name}")
    for e in exs[:2]:
        print(f"[{e['label']}] PREFIKS: {e['prefix']!r}\n    KONTYNUACJA: {e['continuation'][:400]!r}")

### 5b. Augmentacja zbioru treningowego wybranym ustawieniem

In [ ]:
if AUG_PATH.exists():
    log(f"5b: wczytuję {AUG_PATH}")
else:
    prompts = make_prompts(train)
    seen, rows, dropped = set(train_full["text"]), [], {"za krótkie": 0, "duplikaty": 0}
    for c in range(CFG["aug_copies"]):
        log(f"5b: kopia {c + 1}/{CFG['aug_copies']} ({len(prompts)} tekstów, {BEST_SETTING})")
        for g in generate(lm, prompts, GEN_SETTINGS[BEST_SETTING], seed=SEED + c):
            if g["n_cont"] < MIN_CONT:
                dropped["za krótkie"] += 1
            elif g["text"] in seen:
                dropped["duplikaty"] += 1
            else:
                seen.add(g["text"])
                rows.append({"text": g["text"], "label": g["label"], "setting": BEST_SETTING, "copy": c, "n_cont": g["n_cont"], "prefix": g["prefix"]})
    pd.DataFrame(rows).to_json(AUG_PATH, orient="records", lines=True, force_ascii=False)
    (ART / "aug-dropped.json").write_text(json.dumps(dropped, ensure_ascii=False))

aug = pd.read_json(AUG_PATH, lines=True)
dropped = json.loads((ART / "aug-dropped.json").read_text()) if (ART / "aug-dropped.json").exists() else {}
aug["clf_pred"] = classify(clf, list(aug["text"]))
print(f"Wygenerowano: {len(aug) + sum(dropped.values())}, odrzucono: {dropped}, zostało: {len(aug)}")
print(f"Zgodność etykiety całej augmentacji (klasyfikator r={R_AUG}): {(aug['clf_pred'] == aug['label']).mean():.3f}")
display(pd.crosstab(aug["label"], aug["clf_pred"], normalize="index").round(2).rename_axis(index="etykieta promptu", columns="predykcja klasyfikatora"))
for _, row in aug.sample(min(6, len(aug)), random_state=0).iterrows():
    print(f"[{row['label']} -> klasyfikator: {row['clf_pred']}] {row['text'][:300]!r}\n")

del lm, clf
free_gpu()

### 5c. LoRA `r=8` na `train + aug` vs `r=8` z 3a (ten sam test)

In [ ]:
train_aug_ds = make_dataset(pd.concat([train, load_aug(AUG_NAME)]), tok, MAX_LEN)
seq_aug = run_seq(R_AUG, f"seq-r{R_AUG}-aug", train_aug_ds)
aug_vs = pd.DataFrame([
    {"model": f"LoRA r={R_AUG}", "train": len(train_ds), **{k: seq[R_AUG][k] for k in ("dev_f1", "test_accuracy", "test_f1", "train_time_s")}},
    {"model": f"LoRA r={R_AUG} + aug GPT-2", "train": len(train_aug_ds), **{k: seq_aug[k] for k in ("dev_f1", "test_accuracy", "test_f1", "train_time_s")}},
])
px.bar(aug_vs.melt(id_vars="model", value_vars=["test_accuracy", "test_f1"], var_name="metryka", value_name="wynik"),
       x="model", y="wynik", color="metryka", barmode="group", text_auto=".3f",
       title=f"5c. Zbiór testowy: LoRA r={R_AUG} bez augmentacji vs z augmentacją GPT-2").show()
aug_vs

**Wnioski 5** _(do uzupełnienia po pełnym runie)_:
- Jak hiperparametry wpływają na teksty? Czy potwierdza się oczekiwany kompromis: niska `T` / małe `k` to płynność i powtórzenia, wysoka `T` / brak obcinania to różnorodność i chaos?
- Które ustawienie wybrała reguła i czy zgadza się to z oceną „na oko” przykładów?
- Ile wygenerowanych tekstów zachowuje etykietę? Które klasy najczęściej „przechodzą” w inne (tabela krzyżowa)? Jak to się ma do zgodności na samym prefiksie?
- Czy augmentacja pomogła względem `r=8` bez augmentacji? Jaki był koszt (rozmiar zbioru, czas)? Porównanie z `[MASK]` u BERT-a (+0,007 F1).

## Podsumowanie: GPT-2 vs HerBERT

Wyniki HerBERT-a przepisane z `../bert/bert.ipynb` (pełny trening, ten sam test). **W trybie pilota porównanie jest tylko orientacyjne.**

In [ ]:
BERT = {  # z bert.ipynb (osoba A)
    "seq": {0: (0.613415, 0.533046), 4: (0.886585, 0.870244), 8: (0.880488, 0.867788), 16: (0.887805, 0.871723)},
    "tok": {0: (0.970896, 0.134310), 4: (0.975553, 0.237671), 8: (0.983702, 0.387632), 16: (0.975553, 0.317180)},
    "silhouette": {0: -0.008246, 4: 0.471564, 8: 0.496493, 16: 0.510519},
    "aug": {"bez": (0.880488, 0.867788), "z aug": (0.892683, 0.874908)},
}
sil = dict(zip(silhouette["model"], silhouette["silhouette (cosine, 1024D)"]))
rows = []
for r in CFG["ranks"]:
    rows.append({
        "r": name_of(r),
        "tekst F1 BERT": BERT["seq"][r][1], "tekst F1 GPT-2": seq[r]["test_f1"],
        "tokeny F1 BERT": BERT["tok"][r][1], "tokeny F1 GPT-2": tokres[r]["test_f1"],
        "silhouette BERT": BERT["silhouette"][r], "silhouette GPT-2": sil["przed: GPT-2" if r == 0 else f"po: LoRA r={r}"],
        "params BERT (tekst)": {0: 3076, 4: 150532, 8: 297988, 16: 592900}[r], "params GPT-2 (tekst)": seq[r]["trainable_params"],
    })
summary = pd.DataFrame(rows).set_index("r")
summary.loc[f"r={R_AUG} + augmentacja", ["tekst F1 BERT", "tekst F1 GPT-2"]] = [BERT["aug"]["z aug"][1], seq_aug["test_f1"]]
display(summary.round(3))

long = summary.reset_index().melt(id_vars="r", value_vars=[c for c in summary.columns if "F1" in c or "silhouette" in c])
long[["miara", "model"]] = long["variable"].str.rsplit(" ", n=1, expand=True)
px.bar(long.dropna(), x="r", y="value", color="model", facet_col="miara", barmode="group", text_auto=".2f",
       title=f"BERT vs GPT-2 (tryb: {MODE})").update_yaxes(matches=None, showticklabels=True).show()

times = pd.DataFrame(
    [{"eksperyment": p.parent.name, "min": json.loads(p.read_text()).get("wall_time_s", 0) / 60} for p in sorted(ART.glob("*/result.json"))]
)
print(f"Czas całego notebooka: {(time.time() - T0) / 60:.1f} min (bez wczytanych z cache)")
times.round(1)

**Wnioski końcowe** _(do uzupełnienia po pełnym runie)_:
- Który model jest lepszy w klasyfikacji tekstu, a który w klasyfikacji tokenów, i czy różnicę da się wyjaśnić architekturą (dwukierunkowy enkoder vs przyczynowy dekoder)?
- Czy strojenie porządkuje przestrzeń tak samo w obu modelach (silhouette, kształt skupisk)?
- Która augmentacja (`[MASK]` vs generacja) dała więcej, a która jest „bezpieczniejsza” dla etykiet?
- Koszt: parametry adaptera, czas treningu (GPT-2 ok. 3× wolniejszy).